# California / Hawaii NFDRS Normals 2016–2025 — by RAWS, PSA and FDRA

Builds the "what's normal" tables for the CA Fire Weather Experience Builder app.

- **Stations:** the RAWS in the live *CA NFDRS 3.0 (FEMS)* layer that the app displays (California + Hawaii),
  so normals line up 1:1 with the live values: same stations, same PSA assignments, same fuel model (Y).
- **Data:** FEMS daily NFDR summaries, **observed** values only, 2016-01-01 → 2025-12-31.
- **PSA values:** the daily mean of the PSA's key RAWS (fuel model Y, same as the live layer).
- **FDRA values (national forests):** the daily mean of each FDRA's **FDOP stations and fuel model from
  the 2026 Forest Thresholds sheet** (`fdra_station_membership_2026.csv`). ENF 335 has separate peak /
  non-peak station sets, so it gets two units.
- **Indices:** ERC, IC, BI, SC, FM-1, FM-10, FM-100, FM-1000, KBDI, GSI, Woody FM, Herb FM.

**Outputs** (in `normals/output/`), ready to add to ArcGIS Online as hosted tables:

| File | One row per | Use in the app |
|---|---|---|
| `ca_normals_annual_2016_2025.csv` | station / PSA / FDRA | percentile thresholds (p3 … p97) and mean for every index |
| `ca_normals_doy_2016_2025.csv` | PSA / FDRA × day of year | normal band (p10–p90, median, mean) on the season charts |
| `ca_normals_stations_2016_2025.csv` | station | station → PSA → FDRA lookup |
| `ca_fdra_stations_2016_2025.csv` | FDRA | which RAWS feed each FDRA |

Run top to bottom. The download (≈ 2,000 station-years) is cached in `normals/fems_raw/`; if it stops,
just re-run and it picks up where it left off.


In [ ]:
# 1 — Install packages (once)
import subprocess, sys
for pkg in ['pandas', 'numpy', 'requests', 'shapely', 'pyshp', 'pyproj', 'matplotlib', 'python-dotenv']:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'], capture_output=True, text=True)
    print(('OK  ' if r.returncode == 0 else 'FAIL') + f' {pkg}')


In [ ]:
# 2 — Settings  (edit these)
from pathlib import Path
START_YEAR = 2016
END_YEAR   = 2025

HERE     = Path.cwd()                       # run from the repo's normals/ folder
RAW_DIR  = HERE / 'fems_raw'                # cached FEMS downloads (git-ignored)
OUT_DIR  = HERE / 'output'

# FDRA station lists (2026 Forest Thresholds sheet → align_fdras_to_sheet.py output)
FDRA_MEMBERSHIP = HERE / 'fdra' / 'fdra_station_membership_2026.csv'

INCLUDE_STATION_DOY = False                 # day-of-year normals per station (large file)
WORKERS = 8                                 # parallel FEMS requests


In [ ]:
# 3 — FEMS credentials
# Put these two lines in a file named .env in the repo root (it is git-ignored):
#   FEMS_API_KEY=your_key
#   FEMS_USERNAME=your.email@usda.gov
import os, getpass
from dotenv import load_dotenv
load_dotenv(HERE.parent / '.env'); load_dotenv(HERE / '.env')
FEMS_API_KEY  = os.getenv('FEMS_API_KEY')  or getpass.getpass('FEMS API key: ')
FEMS_USERNAME = os.getenv('FEMS_USERNAME') or input('FEMS username (email): ')
print('FEMS credentials loaded for', FEMS_USERNAME)


In [ ]:
# 4 — Stations and PSAs from the live CA NFDRS 3.0 layer
import logging, importlib
logging.basicConfig(level=logging.INFO, format='[%(asctime)s] %(message)s', datefmt='%H:%M:%S', force=True)
import ca_normals as cn; importlib.reload(cn)

stations = cn.load_ca_stations()
psas     = cn.load_ca_psas()
print(stations.groupby('GACC')['PSA'].nunique().rename('PSAs').to_frame().join(
      stations.groupby('GACC').size().rename('RAWS')))
stations.head()


In [ ]:
# 5 — FDRA FDOP stations (national forests, 2026 sheet)
fdra_mem, fdra_pairs = (None, None)
if FDRA_MEMBERSHIP.exists():
    fdra_mem, fdra_pairs = cn.load_fdra_membership(FDRA_MEMBERSHIP)
    print(len(fdra_mem), 'FDRAs;', fdra_pairs.FuelModel.value_counts().to_dict(), 'station × fuel-model pairs')
else:
    print('No FDRA membership file — FDRA normals skipped. Put fdra_station_membership_2026.csv in normals/fdra/.')

# everything to download: live-layer stations (fuel model Y) + FDRA FDOP stations (their fuel model)
downloads = stations[['stationId', 'FuelModel']]
if fdra_pairs is not None:
    downloads = pd.concat([downloads, fdra_pairs], ignore_index=True).drop_duplicates()
print('Station × fuel-model pairs to pull:', len(downloads))


In [ ]:
# 6 — Download FEMS history (resumable; ~20–40 min the first time)
failed = cn.download_history(downloads, START_YEAR, END_YEAR, RAW_DIR, FEMS_API_KEY, FEMS_USERNAME, WORKERS)
print('Failed station-years:', len(failed), '(re-run this cell to retry)' if failed else '')


In [ ]:
# 7 — Load observed history and check coverage
hist = cn.load_history(downloads, START_YEAR, END_YEAR, RAW_DIR)
cov = (hist.assign(year=hist['date'].dt.year)
           .groupby('year').agg(stations=('stationId', 'nunique'), station_days=('erc', 'count')))
display(cov)
low = (hist.groupby(['stationId', 'fm'])['erc'].count() / (365.25 * (END_YEAR - START_YEAR + 1)))
print(f'Stations with < 50% of days: {(low < 0.5).sum()}  →', low[low < 0.5].sort_values().head(15).round(2).to_dict())


In [ ]:
# 8 — Aggregate and compute normals
import pandas as pd
membership = {
    'STATION': {str(s): [s] for s in stations['stationId']},          # fuel model Y
    'PSA':     stations.groupby('PSA')['stationId'].apply(list).to_dict(),
}
if fdra_mem:
    membership['FDRA'] = fdra_mem                                       # per-FDRA fuel model

daily = pd.concat([cn.unit_daily(hist, m, ut) for ut, m in membership.items()], ignore_index=True)
annual = cn.annual_normals(daily)

doy_src = daily if INCLUDE_STATION_DOY else daily[daily['unit_type'] != 'STATION']
doy = cn.doy_normals(doy_src, plot_year=END_YEAR + 1)
print(annual.groupby('unit_type').size().to_dict(), '| DOY rows:', len(doy))
annual[annual['unit_type'] == 'PSA'][['unit_id', 'n_days', 'erc_mean', 'erc_p90', 'erc_p97',
                                     'fm100_p3', 'fm100_p10', 'ic_p90', 'sc_p90']].head(12)


In [ ]:
# 9 — Write outputs
files = cn.write_outputs(OUT_DIR, START_YEAR, END_YEAR, stations, None, annual, doy)
for k, f in files.items():
    print(f'{k:9s} {f}')


In [ ]:
# 10 — Quick look: ERC and FM-100 normals for one PSA
import matplotlib.pyplot as plt
PSA = 'SC10'
d = doy[(doy.unit_type == 'PSA') & (doy.unit_id == PSA)]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, k in zip(axes, ['erc', 'fm100']):
    ax.fill_between(d.plot_date, d[f'{k}_p10'], d[f'{k}_p90'], alpha=.25, label='p10–p90')
    ax.plot(d.plot_date, d[f'{k}_p50'], lw=1.5, label='median')
    ax.set_title(f'{PSA} {k.upper()} normals {START_YEAR}–{END_YEAR}'); ax.legend()
plt.tight_layout(); plt.show()


## Add the tables to ArcGIS Online (NIFC)

For `ca_normals_annual_2016_2025.csv` and `ca_normals_doy_2016_2025.csv` (and the two lookup CSVs):

1. **Content → New item → Your device**, choose the CSV.
2. Choose **Add … and create a hosted table** (no location). Keep the field types it suggests;
   make sure `plot_date` is a **Date** field.
3. Title them `CA NFDRS Normals 2016–2025 – Annual` and `CA NFDRS Normals 2016–2025 – Day of Year`, folder
   *GACC Fire Weather*, tag `ca-fire-wx`.

These are static. Re-run this notebook and overwrite the tables (item → **Update data → Overwrite**) only
when you change the period or the station set.
